## Transformer Model for translation English -> Finnish.

In [ ]:
import keras
import tensorflow as tf
import numpy as np
from keras import layers
from keras import ops

2025-04-24 21:48:15.452244: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-24 21:48:15.452851: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-24 21:48:15.455471: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-24 21:48:15.462117: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745520495.472416   37616 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745520495.47

In [2]:
text_file = "fin-eng/fin.txt"

with open(text_file, encoding='utf-8') as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    english, finnish, rest = line.split("\t")
    finnish = "[start] " + finnish + " [end]"
    text_pairs.append((english, finnish))

print(text_pairs[:10])

[('Go.', '[start] Mene. [end]'), ('Hi.', '[start] Moro! [end]'), ('Hi.', '[start] Terve. [end]'), ('Run!', '[start] Juokse! [end]'), ('Run!', '[start] Juoskaa! [end]'), ('Run.', '[start] Juokse. [end]'), ('Who?', '[start] Kuka? [end]'), ('Wow!', '[start] Mahtavaa! [end]'), ('Wow!', '[start] Siistiä! [end]'), ('Wow!', '[start] Vau! [end]')]


In [3]:
import random

random.seed(42)
random.shuffle(text_pairs)

train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

total_size = len(text_pairs)
train_size = int(total_size * train_ratio)
val_size = int(total_size * val_ratio)

train_pairs = text_pairs[:train_size]
val_pairs = text_pairs[train_size:train_size+val_size]
test_pairs = text_pairs[train_size+val_size:]

print(f"Total pairs: {total_size}")
print(f"Training pairs: {len(train_pairs)}")
print(f"Validation pairs: {len(val_pairs)}")
print(f"Testing pairs: {len(test_pairs)}")

Total pairs: 72258
Training pairs: 57806
Validation pairs: 7225
Testing pairs: 7227


In [4]:
train_ds = tf.data.Dataset.from_tensor_slices(train_pairs).batch(64)
val_ds = tf.data.Dataset.from_tensor_slices(val_pairs).batch(64)
test_ds = tf.data.Dataset.from_tensor_slices(test_pairs).batch(64)

vocab_size = 10000
sequence_length = 250

source_vectorizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length,
)

target_vectorizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length,
)

train_ds_vectorized = train_ds.map(lambda pair: (source_vectorizer(pair[0]), target_vectorizer(pair[1])))
val_ds_vectorized = val_ds.map(lambda pair: (source_vectorizer(pair[0]), target_vectorizer(pair[1])))
test_ds_vectorized = test_ds.map(lambda pair: (source_vectorizer(pair[0]), target_vectorizer(pair[1])))



2025-04-24 21:48:16.632043: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim
        )
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, inputs):
        length = ops.shape(inputs)[-1]
        positions = ops.arange(0, length, 1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return ops.not_equal(inputs, 0)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "sequence_length": self.sequence_length,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config